# EDGAR Financial Sentiment — Part 4: QLoRA, Step by Step

Adapted from **Assignment 3** (QLoRA), retargeted to finance.

Parts 2–3 fine-tuned *small* models in full. Here we adapt a **7-billion-parameter** model, `mistralai/Mistral-7B-Instruct-v0.3`, on the same Financial PhraseBank task — on a single T4. The point of this notebook is **not** to copy a QLoRA recipe; it's to *understand and build* it. You'll write the quantization and LoRA configs yourself, and every step is **instrumented** so you can see exactly what each choice does to **memory**, **trainable parameters**, and **accuracy**.

You keep the `Dataset` and the train/test loops from Parts 2–3 (already your practice). The **new** concept — the QLoRA configuration — is the part you build here.

> **Run in Google Colab with a T4 GPU.** Gated Mistral model (HF login below).

## 0. Why QLoRA? The problem we're solving

Suppose you want to fine-tune Mistral-7B. **Full** fine-tuning has to hold, all at once:

| what | size (fp16) |
|---|---|
| the weights | ~14 GB |
| the gradients (one per trainable weight) | ~14 GB |
| Adam optimizer state (2 per trainable weight) | ~28 GB |
| **total** | **~56 GB** |

A T4 has **16 GB**. So full fine-tuning is simply off the table. **QLoRA** removes the wall with two ideas, and *each maps to one config object you'll write below:*

1. **Q — quantize the frozen base to 4-bit** (`BitsAndBytesConfig`). The ~14 GB of weights become ~3.5 GB. The base is frozen, so it never needs gradients or optimizer state.
2. **LoRA — train tiny low-rank adapters instead of the weights** (`LoraConfig`). Only ~0.1–1% of the parameters are trainable, so the gradient + optimizer memory shrinks from ~42 GB to a fraction of a GB.

The rest of this notebook builds those two pieces and **measures** the payoff at each step. (This mirrors Assignment 3, which had you set the quantization config and inspect GPU memory yourself.)

## 1. Setup

In [1]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets

In [2]:
import random, io, zipfile, requests
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset as HFDataset, ClassLabel

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

Device: cuda


### Hugging Face login (Mistral is gated)
Accept the license at <https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3>, make a token, log in:

In [4]:
from huggingface_hub import login
login()   # paste your HF token when prompted

## 2. Config & seeds

Note: the **LoRA hyperparameters live in the config you'll write in Section 7**, not here — that's the part you're learning.

In [5]:
torch.manual_seed(10); random.seed(10); np.random.seed(10)

MODEL_ID        = 'mistralai/Mistral-7B-Instruct-v0.3'
NUM_CLASSES     = 3
max_len         = 64
batch_size      = 4       # small: 4-bit base + gradient checkpointing keep this comfortable on a T4
eval_batch_size = 8
lr              = 2e-4    # typical LoRA learning rate
TRAIN_STEPS     = 500     # ~1 epoch over PhraseBank at batch_size 4; raise for more

## 3. Load the Financial PhraseBank benchmark

Same data + stratified split as Parts 2–3, from the raw zip.

In [6]:
_URL = 'https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip'
_z = zipfile.ZipFile(io.BytesIO(requests.get(_URL, timeout=120).content))
_member = next(n for n in _z.namelist() if n.endswith('Sentences_AllAgree.txt'))
_raw = _z.read(_member).decode('latin-1')

_label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
_rows = []
for _line in _raw.splitlines():
    _line = _line.strip()
    if not _line:
        continue
    _sent, _lab = _line.rsplit('@', 1)
    _rows.append({'sentence': _sent, 'label': _label_map[_lab.strip()]})
_df = pd.DataFrame(_rows)

pb_full = HFDataset.from_pandas(_df).cast_column('label', ClassLabel(names=['negative', 'neutral', 'positive']))
pb = pb_full.train_test_split(test_size=0.2, seed=10, stratify_by_column='label')
pb_train, pb_test = pb['train'], pb['test']
print('Train size:', len(pb_train), '| Test size:', len(pb_test))

Casting the dataset:   0%|          | 0/2264 [00:00<?, ? examples/s]

Train size: 1811 | Test size: 453


## 4. The memory wall, in numbers

Before loading anything, let's make the problem concrete.

In [7]:
B = 7  # billion parameters (Mistral-7B)
print('Storing the 7B weights alone:')
for name, bytes_per in [('fp32', 4), ('fp16', 2), ('int8', 1), ('4-bit nf4', 0.5)]:
    print(f'  {name:10s} ~ {B*bytes_per:5.1f} GB')
print()
print('FULL fine-tuning in fp16 needs weights + grads + Adam states:')
print('  ~14 (weights) + ~14 (grads) + ~28 (Adam) = ~56 GB   -> a T4 has 16 GB, so: impossible')
print('QLoRA needs:')
print('  ~3.5 GB (4-bit frozen weights) + grads/optimizer for ONLY the tiny adapters (~0.1 GB)')

Storing the 7B weights alone:
  fp32       ~  28.0 GB
  fp16       ~  14.0 GB
  int8       ~   7.0 GB
  4-bit nf4  ~   3.5 GB

FULL fine-tuning in fp16 needs weights + grads + Adam states:
  ~14 (weights) + ~14 (grads) + ~28 (Adam) = ~56 GB   -> a T4 has 16 GB, so: impossible
QLoRA needs:
  ~3.5 GB (4-bit frozen weights) + grads/optimizer for ONLY the tiny adapters (~0.1 GB)


In [8]:
# Two instruments we'll reuse to SEE the effect of each step.
def show_gpu_mem(label=''):
    torch.cuda.empty_cache()
    gb = torch.cuda.memory_allocated() / 1024**3
    tag = f' ({label})' if label else ''
    print(f'GPU memory allocated{tag}: {gb:.2f} GB')

def trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 5. Step 1 — Quantize the base to 4-bit (the **Q**)  &larr; **your code**

`BitsAndBytesConfig` tells `from_pretrained` how to shrink the weights as it loads them. Set these fields:
- `load_in_4bit=True` — store base weights in 4 bits instead of 16.
- `bnb_4bit_quant_type='nf4'` — *normal-float-4*, a 4-bit grid designed for neural-net weight distributions (better than plain int4).
- `bnb_4bit_use_double_quant=True` — also quantize the per-block quantization constants (a little extra saving).
- `bnb_4bit_compute_dtype=torch.float16` — the actual matmuls run in fp16 (the T4 has no bf16).

Write the config below.

In [9]:
### YOUR CODE HERE
# Build bnb_config = BitsAndBytesConfig(...) with the four fields described above:
#   load_in_4bit, bnb_4bit_quant_type, bnb_4bit_use_double_quant, bnb_4bit_compute_dtype
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_wuant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)
### END YOUR CODE

### Load the model in 4-bit
Mistral has no pad token, so we reuse its EOS token, and pad on the **right** — the classification head finds the last non-pad token assuming right-padding. Watch the memory print: this is your 4-bit base **measured**.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_CLASSES,
    quantization_config=bnb_config, device_map={'': 0},   # <- uses YOUR bnb_config
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
show_gpu_mem('after loading 4-bit base + fresh classification head')

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


GPU memory allocated (after loading 4-bit base + fresh classification head): 3.61 GB


#### ✍️ Reflect — what did 4-bit buy?
- Compare the measured `GPU memory allocated (after loading...)` to the **fp16** row of the arithmetic in Section 4. Roughly how much did 4-bit save?
A: The model with 7 billion parameters in float 32 would have taken 28 GB of RAM. We nf4 to quantize to 4 bit weights, and save 24.4 GB of memory. This allows us to use a higher quality model for the amount of memory we have on the T4.
- Why is it OK to store the *base* in low precision, while training the adapters in higher precision?
A: The base weights are not changing and we do not lose a lot of precision by changing the data type from 32 to 4. The quantization does scale up the numbers temporariliy to 16 speed up calculation.
The trainable parameters maintain greater precision so the model can learn about our training with a greater degree of accuracy for the limit number of trainable weights available.

## 6. Step 2 — Freeze the base & prep for k-bit training

`prepare_model_for_kbit_training` freezes the quantized base, turns on **gradient checkpointing** (recompute activations in the backward pass instead of storing them — trades compute for memory), and enables input gradients so the adapters can train. Watch the trainable-parameter count collapse: after this, almost nothing is trainable yet — we add the trainable bits in Step 3.

In [12]:
print(f'trainable params BEFORE prepare: {trainable_params(model):,}')
model = prepare_model_for_kbit_training(model)
print(f'trainable params AFTER  prepare: {trainable_params(model):,}   '
      '(base frozen; gradient checkpointing + input grads enabled)')

trainable params BEFORE prepare: 134,496,256
trainable params AFTER  prepare: 0   (base frozen; gradient checkpointing + input grads enabled)


## 7. Step 3 — Inject LoRA adapters (the **LoRA**)  &larr; **your code**

LoRA freezes each targeted weight matrix `W` and learns a small **low-rank** update `B·A` of rank `r`, so the layer computes `W x + B(A x)`. Only `A` and `B` train. The knobs:
- `r` — the rank: how big the low-rank update is. **Bigger r = more capacity and more trainable params.**
- `lora_alpha` — a scaling factor; the update is multiplied by `alpha / r`.
- `lora_dropout` — dropout on the adapter inputs (regularization).
- `bias='none'` — don't train biases.
- `task_type=TaskType.SEQ_CLS` — sequence classification (also keeps the new head trainable).
- `target_modules` — **which** Linear layers get adapters. We adapt the attention projections `['q_proj','k_proj','v_proj','o_proj']`.

Write the `LoraConfig` and wrap the model with `get_peft_model`.

In [13]:
### YOUR CODE HERE
# 1. lora_config = LoraConfig(...) with:
#       r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
#       task_type=TaskType.SEQ_CLS,
#       target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
# 2. wrap the base:  model = get_peft_model(model, lora_config)
...
### END YOUR CODE

Ellipsis

In [14]:
model.print_trainable_parameters()   # trainable vs total, and the % -- the headline of QLoRA
show_gpu_mem('after attaching LoRA adapters')
loss_fn = nn.CrossEntropyLoss()

AttributeError: 'MistralForSequenceClassification' object has no attribute 'print_trainable_parameters'

### How do trainable params scale with rank `r`?
Each adapted Linear of shape `(in, out)` adds `r·(in + out)` parameters (`A` is `r×in`, `B` is `out×r`). So the trainable count scales **linearly with `r`** and with how many modules you target. Here it is, computed directly from the model:

In [ ]:
def lora_param_count(model, r, targets=('q_proj', 'k_proj', 'v_proj', 'o_proj')):
    n = 0
    for name, module in model.named_modules():
        if any(name.endswith(t) for t in targets) and hasattr(module, 'in_features'):
            n += r * (module.in_features + module.out_features)
    return n

print('LoRA trainable params vs rank r (targets: q,k,v,o_proj):')
for r in [4, 8, 16, 32, 64]:
    n = lora_param_count(model, r)
    print(f'  r={r:>3}: {n:>12,} params   (~{100*n/7e9:.3f}% of 7B)')

#### ✍️ Reflect — why does this fit?
- What % of parameters are trainable (from `print_trainable_parameters`)? _…_
- Gradients + optimizer state are stored only for trainable params. Roughly how much memory does that save vs. full fine-tuning? _…_
- From the table above: if you doubled `r`, what happens to the adapter param count? Is memory the reason to keep `r` small here? _…_

## 8. Build the `Dataset`  &larr; **your code** *(kept from your Parts 2–3 practice)*

Same as before: tokenize, right-pad, keep the class index. CPU tensors; the loops move batches to the GPU.

In [ ]:
class ClassificationData(Dataset):
    """Tokenize (sentence, label) rows for Mistral sequence classification.

    Right-padding (the classification head locates the last non-pad token that way).
    Per row store 'input_ids', 'attention_mask' (max_len,) and 'label' (scalar int64),
    all on CPU.
    """
    def __init__(self, hf_split, tokenizer, max_len):
        self.data = []
        for ex in hf_split:
            ### YOUR CODE HERE
            # 1. tokenize ex['sentence'] with: max_length=max_len, truncation=True,
            #    padding='max_length', return_tensors='pt'
            # 2. append a CPU dict to self.data with 'input_ids', 'attention_mask'
            #    (each .squeeze(0)) and 'label' = torch.tensor(ex['label'], dtype=torch.int64)
            ...
            ### END YOUR CODE

    def __len__(self):
        ### YOUR CODE HERE
        ...
        ### END YOUR CODE

    def __getitem__(self, index):
        ### YOUR CODE HERE
        ...
        ### END YOUR CODE

### Sanity-check your `Dataset`
The first token should be Mistral's beginning-of-sequence token `<s>`.

In [ ]:
_probe = ClassificationData(pb_train, tokenizer, max_len)
print('Examples built:', len(_probe))
_ex = _probe[0]
print('input_ids shape:', tuple(_ex['input_ids'].shape), '| real tokens:', int(_ex['attention_mask'].sum()))
print('first token:', repr(tokenizer.decode([int(_ex['input_ids'][0])])), '(Mistral BOS)')
print('label:', int(_ex['label']))

## 9. Train and test loops  &larr; **your code** *(kept)*

Same as Part 3, with one difference you've already met: `model` is the HF classifier, so the loop calls `model(input_ids=..., attention_mask=...)` and reads `out.logits`.

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer, reporting_interval=50, steps=None):
    """One pass of QLoRA fine-tuning.

    - model.train(); track running loss + batch count
    - for each batch (stop after `steps`):
        - move the batch dict to device
        - out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        - logits = out.logits  -> (batch, num_classes); targets = batch['label']
        - zero_grad, CrossEntropyLoss, backward, optimizer.step()
        - accumulate loss; print running avg every `reporting_interval` batches
    """
    ### YOUR CODE HERE
    # follow the steps in the docstring above
    ...
    ### END YOUR CODE

In [ ]:
def test_loop(dataloader, model, loss_fn, steps=None):
    """Evaluate accuracy: argmax(logits) == label.

    - model.eval(); torch.no_grad()
    - per batch (stop after `steps`): move to device; out = model(input_ids=..., attention_mask=...);
      logits = out.logits; add loss; count correct (argmax == label) and total
    - print accuracy + avg loss; return accuracy in percent
    """
    ### YOUR CODE HERE
    ...
    ### END YOUR CODE

## 10. Run QLoRA fine-tuning — and *see* accuracy move

First we evaluate **before** training. The classification head is randomly initialized and the base is frozen, so we expect ~chance (≈33% over 3 classes). Then we train, and re-evaluate. Because the base never changes, the entire jump is what the **trainable LoRA adapters + head** learned — that's the accuracy effect of the config you built.

In [ ]:
print('Accuracy BEFORE training (random head, frozen 4-bit base):')
_baseline_loader = DataLoader(ClassificationData(pb_test, tokenizer, max_len), batch_size=eval_batch_size, shuffle=False)
_ = test_loop(_baseline_loader, model, loss_fn, steps=20)

In [ ]:
train_loader = DataLoader(ClassificationData(pb_train, tokenizer, max_len),
                          batch_size=batch_size, shuffle=True, drop_last=True)
test_loader  = DataLoader(ClassificationData(pb_test, tokenizer, max_len),
                          batch_size=eval_batch_size, shuffle=False)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)

print('QLoRA fine-tuning Mistral-7B...')
train_loop(train_loader, model, loss_fn, optimizer, steps=TRAIN_STEPS)
print('Evaluating...')
acc = test_loop(test_loader, model, loss_fn)
print(f'\n================ RESULT ================')
print(f'Mistral-7B (QLoRA) test accuracy: {acc:.1f}%')

## 11. (Optional) Does rank `r` change accuracy?

Memory and parameter effects were easy to see live. **Accuracy** is the expensive dimension — comparing two configs means *training twice more*. Flip `RUN_RANK_ABLATION = True` to train a low- and high-rank adapter briefly and compare. It's slow and memory-heavy (it reloads the base), so it's off by default. **Expected on this clean task:** small differences from `r` — capacity isn't the bottleneck — while *which* `target_modules` you adapt usually matters more than `r`. *(If you hit OOM, run this in a fresh runtime.)*

In [ ]:
RUN_RANK_ABLATION = False   # set True to compare ranks (slow: two extra short 7B trainings)

def _train_eval_lora(r, steps=150, eval_steps=20):
    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=NUM_CLASSES, quantization_config=bnb_config, device_map={'': 0})
    m.config.pad_token_id = tokenizer.pad_token_id; m.config.use_cache = False
    m = prepare_model_for_kbit_training(m)
    m = get_peft_model(m, LoraConfig(r=r, lora_alpha=2*r, lora_dropout=0.05, bias='none',
                                     task_type=TaskType.SEQ_CLS,
                                     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']))
    tr = DataLoader(ClassificationData(pb_train, tokenizer, max_len), batch_size=batch_size, shuffle=True, drop_last=True)
    te = DataLoader(ClassificationData(pb_test, tokenizer, max_len), batch_size=eval_batch_size, shuffle=False)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr)
    train_loop(tr, m, loss_fn, opt, steps=steps)
    acc = test_loop(te, m, loss_fn, steps=eval_steps)
    del m, opt; torch.cuda.empty_cache()
    return acc

if RUN_RANK_ABLATION:
    for r in [4, 32]:
        print(f'--- rank r={r} ---')
        print(f'r={r}: test accuracy {_train_eval_lora(r):.1f}%\n')
else:
    print('RUN_RANK_ABLATION is off. Section 10 already showed the before/after-training accuracy delta.')

## 12. What you measured — tying it together

You built QLoRA from its two configs and watched each one pay off:

| Config you wrote | What it changes | How you saw it |
|---|---|---|
| `BitsAndBytesConfig` (4-bit) | **Memory** of the frozen base: ~14 GB → ~3.5 GB | `show_gpu_mem` after load vs. the fp16 arithmetic |
| `prepare_model_for_kbit_training` | Freezes base, enables gradient checkpointing | trainable-param count collapses |
| `LoraConfig` (`r`, `target_modules`) | **Trainable parameters**: <1% of 7B | `print_trainable_parameters` + the r-scaling table |
| training the adapters | **Accuracy**: ~33% → high | before/after eval in Section 10 |

**The big picture:** 4-bit shrinks the *frozen* part you don't train; LoRA shrinks the *trainable* part you do. Together they turn a 56 GB job into a 16 GB one — without either, a 7B doesn't fit a T4. That's the whole reason QLoRA exists.

**Things to try:** add the MLP projections (`gate_proj`/`up_proj`/`down_proj`) to `target_modules` and re-check the trainable count; raise `r`; turn on the Section 11 ablation.

**Next (Track B):** Part 5 — prompt this same model zero-/few-shot with *no* training, as the baseline that asks whether all this fine-tuning was even necessary.